# PG&E Hosting Capacity Calculations

In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import box, MultiLineString
import matplotlib.pyplot as plt

In [2]:
os.environ['PROJ_LIB'] = '/opt/anaconda3/share/proj'

In [3]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

In [4]:
# Read in circuits
circuits_raw = gpd.read_parquet("../../../../../../capstone/electrigrid/data/utilities/pge_updated/pge.parquet").to_crs("EPSG:4326")

In [5]:
# Read in customer usage data
feeder = gpd.read_file("../../../../../../capstone/electrigrid/data/utilities/pge_shapefiles/FeederDetail.shp")

In [6]:
## Unit conversion --> no longer necessary with kW data
# vars = ['GenCapacity_','GenericPVC', 'GenCapacity_no_OpFlex_kW', , 'LoadCapacity_kW']

# for col in vars:
#     circuits_raw[col] = circuits_raw[col] / 1000

In [7]:
circuits_raw = circuits_raw[['FeederId', 'CSV_LineSection','LoadCapacity_kW', 'GenCapacity_kW', 'GenericPVCapacity_kW', 'GenCapacity_no_OpFlex_kW', 'GenericCapacity_no_OpFlex_kW', 'geometry']]
circuits_raw.head(2)

,FeederId,CSV_LineSection,LoadCapacity_kW,GenCapacity_kW,GenericPVCapacity_kW,GenCapacity_no_OpFlex_kW,GenericCapacity_no_OpFlex_kW,geometry
0,254611104,5724447,NaN,0.0,0.0,0.0,0.0,"MULTILINESTRING ((-120.09424 36.99662, -120.09..."
1,254611104,3485801,0.0,0.0,0.0,0.0,0.0,"MULTILINESTRING ((-120.10806 37.01024, -120.10..."


### Calculate customer usage percentages

In [8]:
# Calculate total customers per feeder
feeder['total'] = feeder['ResCust'] + feeder['ComCust'] + feeder['IndCust'] + feeder['AgrCust'] + feeder['OthCust']

# Percentage of each customer type
feeder['perc_res'] = feeder['ResCust'] / feeder['total'] * 100
feeder['perc_com'] = feeder['ComCust'] / feeder['total'] * 100
feeder['perc_ind'] = feeder['IndCust'] / feeder['total'] * 100
feeder['perc_agr'] = feeder['AgrCust'] / feeder['total'] * 100
feeder['perc_oth'] = feeder['OthCust'] / feeder['total'] * 100

In [9]:
feeder.head(2)

feeder = feeder.rename(columns = {'FeederID':'FeederId'})

In [10]:
# Keep only rows of interest
feeder = feeder[['FeederId','perc_res', 'perc_com', 'perc_ind', 'perc_agr', 'perc_oth']]

### Join customer usage to feeder IDs

In [11]:
circuits = pd.merge(circuits_raw, feeder, on = "FeederId")

In [12]:
circuits.head(2)

,FeederId,CSV_LineSection,LoadCapacity_kW,GenCapacity_kW,GenericPVCapacity_kW,GenCapacity_no_OpFlex_kW,GenericCapacity_no_OpFlex_kW,geometry,perc_res,perc_com,perc_ind,perc_agr,perc_oth
0,254611104,5724447,NaN,0.0,0.0,0.0,0.0,"MULTILINESTRING ((-120.09424 36.99662, -120.09...",76.08802,12.762836,4.303178,6.356968,0.488998
1,254611104,3485801,0.0,0.0,0.0,0.0,0.0,"MULTILINESTRING ((-120.10806 37.01024, -120.10...",76.08802,12.762836,4.303178,6.356968,0.488998


# Calculating Hosting Capacity for a Feederline with Values!

## Step 0: Load in the other necessary data

In [13]:
# PG&E shapefile
utility_ter = gpd.read_file("../../../../../../capstone/electrigrid/data/utilities/IOU_shapefiles.geojson")

# filter to only pge 
pge_shape = utility_ter[utility_ter['Acronym'] == 'PG&E'] 

# check the crs
pge_shape = pge_shape.to_crs('EPSG:4326')

In [14]:
# read in building data
buildings = gpd.read_parquet("../../../../../../capstone/electrigrid/data/zillow_complete_units.parquet").to_crs("EPSG:4326")
buildings = gpd.clip(buildings, pge_shape)

In [15]:
# read in the census tract data
census_tracts = gpd.read_file("../../../../../../capstone/electrigrid/data/census/tl_2025_06_tract/tl_2025_06_tract.shp").to_crs('EPSG:4326')
census_tracts = gpd.clip(census_tracts, pge_shape)

## Step 1: Link the homes to the nearest feederline

Each home gets its electricty from a specific feederline. The nearest neighbor is the method chosen. Then to avoid including homes that are outliers and likely gettin electricity from other sources homes that are further than 1km from their assigned feederline are dropped. 

In order to match the workflow below, I will rename some variables first.

#### Change column names to match other utilites

In [16]:
circuits = circuits.rename(columns = {'FeederId':'circuit_id',
                                        'CSV_LineSection':'segment_id',
                                        'LoadCapacity_kW':'load_cap',
                                        'GenCapacity_no_OpFlex_kW':'gencap',
                                        'GenCapacity_kW':'gencap_opflex',
                                        'GenericCapacity_no_OpFlex_kW':'gencap_pv',
                                        'GenericPVCapacity_kW':'gencap_pv_opflex'})

### Step 1a: Multiply generations by residential customer percentage

In [17]:
for col in ['gencap', 'gencap_pv', 'gencap_opflex', 'gencap_pv_opflex', 'load_cap']:
    circuits[col] = circuits[col] * (circuits['perc_res'] / 100)

In [ ]:
# renaming variables :)
clipped = gpd.clip(circuits, pge_shape)
buildings_clipped = buildings
census_clipped = census_tracts

In [ ]:
clipped.head(2)

#### Step 1a: Create line length column

Our data analysis will need length data for each line segment. This value is provided in the SCE data but must be calculated for the SDG&E data. Let's calculate the lengths for each segment of the feederline.

In [ ]:
# change the crs to a projected CRS
clipped = clipped.to_crs("EPSG:3310")
buildings_clipped = buildings_clipped.to_crs("EPSG:3310")
census_clipped = census_clipped.to_crs("EPSG:3310")

## create length column in metres
clipped['length_m'] = clipped.length

In [ ]:
clipped.head(2)

### Step 1b: Complete join

In [ ]:
# index the data
clipped.sindex
buildings_clipped.sindex

# spatial join
buildings_linked = gpd.sjoin_nearest(buildings_clipped, 
                                        clipped, 
                                        how="left", 
                                        lsuffix='_left', 
                                        rsuffix='_right',
                                        distance_col='dist_to_line_m')

### Step 1c: Filter for homes that are less than 1000m away from the nearest feederline
If they are more than 1 km away, we assume they get their power from a different source.

In [ ]:
buildings_linked = buildings_linked[buildings_linked['dist_to_line_m'] < 1000]

#### Step 2: Add census tract ID (`geoid`) to each home through spatial join

These tract IDs should be added to the homes dataframes as they are necessary for further calculations. 


In [ ]:
#buildings_linked = buildings_linked.drop('index__right', axis = 1)
buildings_linked.head(2)

In [ ]:
# rename GEOID column
buildings_linked = buildings_linked.rename(columns = {'GEOID':'geoid',
                                                     'unit':'total_units'}) # remove with new data
census_clipped = census_clipped.rename(columns = {'GEOID':'geoid'})

In [ ]:
# make sure the dataframes have the same crs before performing spatial computations
assert census_clipped.crs == buildings_linked.crs

In [ ]:
buildings_linked = buildings_linked.drop(columns=['geoid', 'index_right'], errors='ignore')
buildings_linked = gpd.sjoin(buildings_linked, census_clipped[['geoid', 'geometry']], how="left", predicate="within")

#### Some homes may intersect two census tracts

We assign homes to the tract that contains more of the house's area, in order to avoid duplicates

In [ ]:
# length of dataframe before
len(buildings_linked)

In [ ]:
## FOR BUILDINGS INTERSECTING MORE THAN ONE CENSUS TRACT CALCULATE WHICH TRACT IT INTERSECTS MORE
# change the crs to a projected crs
buildings_linked = buildings_linked.to_crs("EPSG:6933")
census_clipped = census_clipped.to_crs("EPSG:6933")

# calculate intersection area for each building-parcel pair
buildings_linked["intersection_area"] = (
    buildings_linked.geometry.values
    .intersection(census_clipped.loc[buildings_linked["index_right"]].geometry.values).area)

# keep only the parcel with the largest overlap per building
buildings_linked = (
    buildings_linked
    .sort_values("intersection_area", ascending=False)
    .loc[~buildings_linked.index.duplicated(keep="first")]
    .drop(columns="intersection_area"))

# drop the unnecessary columns
buildings_linked = buildings_linked.drop(columns = ['index_right', 'unit_left'])
buildings_linked = buildings_linked.rename(columns = {'unit_right': 'total_unit'})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

ax.axis('off')

census_clipped.boundary.plot(ax=ax, 
                    color='gray')

clipped.plot(ax=ax, 
                    color='blue')

buildings_linked.plot(ax=ax, color = 'red')

In [ ]:
census_clipped = census_clipped.to_crs('EPSG:3310')

#### Step 3: Calculate the number of homes in each census tract

The number of homes in each census tract is used further in the calculation. Save it as a new column in the homes dataframes. 

In [ ]:
buildings_linked['homes_per_tract'] = buildings_linked.groupby('geoid')['total_units'].transform('sum')

#### Step 4: Calculate the maximum ICA generation capacity for each circuit

Generation capacity varies across the circuit. The maximum is utilized in calculating the hosting capacity. Add the new data to the data frame as a column.


In [ ]:
# save the max ICA generation capacity by the circuit
buildings_linked['max_gen_circuit'] = buildings_linked.groupby('circuit_id')['gencap'].transform('max')

#### Step 5: Calculate the percentage of the length of each segment relative to the entire feederline

First calculate the length of the whole circuit. Then divide each segment’s length by the length of the whole circuit. Save to the dataframe in a new column. 


In [ ]:
# calculate the percentage of the length of each segment relative to the whole dataframe
clipped['percent_length'] = clipped['length_m'] / clipped.groupby('circuit_id')['length_m'].transform('sum')

In [ ]:
buildings_linked.head()

#### Step 6:  Calculate the number of homes connected to each segment



In [ ]:
# aggregate number of homes that connect to each segment
buildings_linked['homes_per_segment'] = buildings_linked.groupby('segment_id')['total_units'].transform('sum')

#### Step 7:  Calculate the total number of homes connected to the whole circuit

In [ ]:
# aggregate number of homes that connect to each circuit
buildings_linked['homes_per_circuit'] = buildings_linked.groupby('circuit_id')['circuit_id'].transform('count')

#### Step 8:  Calculate the percent of homes connected to each segment relative to all of the homes connected to the entire circuit

In [ ]:
buildings_linked['percent_homes_circuit'] = buildings_linked['homes_per_segment'] / buildings_linked['homes_per_circuit']

#### Step 9:  Calculate the weighted generation capacity for each segment of the feederline

Multiply the total generation capacity of the segment by the percent of homes connected to each segment. This calculation undercounts the total generation capacity.

In [ ]:
# weighted generation capacity calculations
buildings_linked['weighted_gencap'] = buildings_linked['gencap'] * buildings_linked['percent_homes_circuit']

#### Step 10:  Calculate the number of homes located within each census tract for each circuit


In [ ]:
buildings_linked['homes_per_tract_circuit'] = buildings_linked.groupby(['circuit_id', 'geoid'])['circuit_id'].transform('count')

#### Step 11:  Calculate the weighted household max hosting capacity for each circuit tract combo (Brockway Eq 8)

Divide the number of homes located within each census tract for each circuit by the total number of homes connected to the whole circuit and multiply that value by the max generation capacity report for each circuit tract combo.


In [ ]:
# calculate the max generation capacity for the circuit tract combo
buildings_linked['max_gen_tract_circuit'] = buildings_linked.groupby(['circuit_id', 'geoid'])['gencap'].transform('max')

In [ ]:
# calculate the weighted household max hosting capacity for each circuit tract combo
buildings_linked['weighted_max_gen_circuit_tract'] = (buildings_linked['homes_per_tract_circuit'] / buildings_linked['homes_per_circuit']) * buildings_linked['max_gen_tract_circuit']

####  Step 12:  Normalize the generation capacity for each circuit polygon combo (Brockway Eq 9)

We normalize the generation capacity because weighting the generation capacity under counts the total number of MW available. 

Divide the maximum hosting capacity from anywhere on the given circuit by the  weighted household max hosting capacity for each circuit tract combo, and then multiply by the  weighted household max hosting capacity for each circuit tract combo. 

Weighted max generation per circuit tract is aggregated to just each circuit in this case.

In [ ]:
buildings_linked['normalized_gen'] = (buildings_linked['max_gen_circuit'] / buildings_linked.groupby('circuit_id')['weighted_max_gen_circuit_tract'].transform('sum')) * buildings_linked['weighted_max_gen_circuit_tract']

#### Step 13:  Adjust the normalized values to the maximum generation (Brockway Eq 10)

For each normalized value for the circuit polygon combos, ensure they are not over the original maximum value for that circuit polygon. Adjust down to the max where necessary.


In [ ]:
buildings_linked['normalized_gen_adj'] = np.where(
    buildings_linked['normalized_gen'] > buildings_linked['max_gen_circuit'],
    buildings_linked['max_gen_circuit'],
    buildings_linked['normalized_gen'])

#### Step 14: Calculate the household hosting capacity (Brockway Eq 11)

Divide the generation capacity for the circuit tracts by the number of homes connected in that circuit tract.

In [ ]:
buildings_linked['hh_gencap'] = (buildings_linked['normalized_gen_adj'] / buildings_linked['homes_per_tract_circuit']) * 1000

In [ ]:
print(buildings_linked.columns.tolist())

## Steps 4-14 on the rest of the generation variables

In [ ]:
# Run the Steps 4-14 pipeline for each remaining cap column
for col in ['gencap_pv', 'gencap_opflex', 'gencap_pv_opflex', 'load_cap']:

    # Step 4: max generation capacity by circuit
    buildings_linked[f'max_{col}_circuit'] = buildings_linked.groupby('circuit_id')[col].transform('max')

    # Step 9: weighted generation capacity
    buildings_linked[f'weighted_{col}'] = buildings_linked[col] * buildings_linked['percent_homes_circuit']

    # Step 11a: max generation capacity per circuit/tract combo
    buildings_linked[f'max_{col}_circuit_tract'] = buildings_linked.groupby(['circuit_id', 'geoid'])[col].transform('max')

    # Step 11b: weighted household max hosting capacity per circuit/tract combo
    buildings_linked[f'weighted_max_{col}_circuit_tract'] = (
        buildings_linked['homes_per_tract_circuit'] / buildings_linked['homes_per_circuit']
    ) * buildings_linked[f'max_{col}_circuit_tract']

    # Step 12: normalize generation capacity
    buildings_linked[f'normalized_{col}'] = (
        buildings_linked[f'max_{col}_circuit'] / buildings_linked[f'weighted_max_{col}_circuit_tract']
    ) * buildings_linked[f'weighted_max_{col}_circuit_tract']

    # Step 13: adjust normalized values down to circuit/tract max where needed
    buildings_linked[f'normalized_{col}_adj'] = np.where(
        buildings_linked[f'normalized_{col}'] > buildings_linked[f'max_{col}_circuit_tract'],
        buildings_linked[f'max_{col}_circuit_tract'],
        buildings_linked[f'normalized_{col}'])

    # Step 14: individual home hosting capacity in KW
    buildings_linked[f'hh_{col}'] = (
        buildings_linked[f'normalized_{col}_adj'] / buildings_linked['homes_per_tract_circuit']
    ) * 1000


### Step 15: Add load capacity (Brockway Eq 12)

In [ ]:
# Add load capacity to generation capacity variables
for col in ['gencap', 'gencap_pv', 'gencap_opflex', 'gencap_pv_opflex']:
    buildings_linked[f'hh_{col}'] = buildings_linked[f'hh_{col}'] + buildings_linked[f'hh_load_cap']

In [ ]:
buildings_linked.head()

In [ ]:
buildings_linked['hh_load_cap'].describe()

In [ ]:
buildings_linked['hh_gencap_opflex'].describe()

In [ ]:
buildings_linked[buildings_linked['hh_gencap_opflex'] < 50000]['hh_gencap_opflex'].plot(kind = "hist")

In [ ]:
buildings_linked[buildings_linked['hh_gencap_pv'] < 30000]['hh_gencap_pv'].plot(kind = "hist")